[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C42_Learning_Theory_Course/04_implicit_bias/04_讲解.ipynb)

# 04 · 隐式偏置（GD 自发挑 max-margin / 最小范数解）

目标：实测过参数化下 GD 在解集中的**隐式偏好**：可分 logistic → max-margin 方向；最小二乘 from-0 → 最小范数解；并量化平坦度与隐式正则。

路线：可分数据 GD 方向→max-margin(cos→1) → 最小二乘 GD-from-0→最小范数(cos=1) → SGD 噪声偏向平坦极小 → early stopping ≈ L2 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊。

> 这些结论可证明、可实测：cosine 相似度 → 1 把『GD 挑了谁』钉死。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
print('numpy', np.__version__)

## 1 · 可分数据：GD 方向收敛到 max-margin（hard-SVM）解

线性可分数据 + logistic 损失。GD 会让 $\|w\|\to\infty$，但**方向** $w/\|w\|$ 收敛到 max-margin 解（Soudry 2018）。我们跑 GD 与一个 hard-margin SVM 求解器，验证两者方向 cosine → 1。

In [ ]:
def make_separable(n, d, seed=2, margin=0.7):
    g = np.random.default_rng(seed)
    w_star = g.standard_normal(d); w_star /= np.linalg.norm(w_star)
    X = g.standard_normal((n, d))
    yv = np.sign(X @ w_star)
    X = X + (yv[:, None] * w_star[None, :]) * margin     # 推开制造间隔
    return X, np.sign(X @ w_star)
X, y = make_separable(30, 5, margin=0.7)
def logistic_gd(X, y, steps=60000, lr=1.0):
    n, d = X.shape; w = np.zeros(d)
    for _ in range(steps):
        p = 1/(1+np.exp(y*(X@w)))        # = sigma(-y w.x)
        w = w + lr*(X.T@(p*y))/n           # -grad
    return w
def hard_svm(X, y, iters=40000, lr=0.05, C=1000.0):
    n, d = X.shape; w = np.zeros(d)
    for t in range(iters):
        viol = (y*(X@w)) < 1
        g = w - C*((X*y[:,None])[viol].sum(0))/n     # 0.5||w||^2 + C*hinge 的次梯度
        w = w - lr/np.sqrt(t+1)*g
    return w
w_gd = logistic_gd(X, y); w_svm = hard_svm(X, y)
cos = (w_gd@w_svm)/(np.linalg.norm(w_gd)*np.linalg.norm(w_svm))
print(f'cos(GD 方向, SVM 方向) = {cos:.4f}  (||w_gd||={np.linalg.norm(w_gd):.1f} 已很大)')
assert cos > 0.9, 'GD 方向应收敛到 max-margin 方向'
print('✅ logistic GD 的方向收敛到 max-margin(hard-SVM)解 —— 隐式 max-margin 偏置被钉死。')

## 2 · 最小二乘：GD-from-0 收敛到最小范数插值解

过参数化最小二乘（$n<d$），解集是仿射空间。最小范数解 $w_{\min}=X^\top(XX^\top)^{-1}y$。验证 GD-from-0 收敛到它（cosine=1、范数相等），因为迭代始终停留在 $X^\top$ 的行空间。

In [ ]:
n, d = 5, 20                       # 过参数化 d > n
Xu = rng.standard_normal((n, d)); yu = rng.standard_normal(n)
w_min = Xu.T @ np.linalg.solve(Xu @ Xu.T, yu)     # 最小范数闭式
assert np.allclose(Xu @ w_min, yu), '最小范数解应插值'
def ls_gd(X, y, steps=200000, lr=0.05):
    n, d = X.shape; w = np.zeros(d)
    for _ in range(steps):
        w = w - lr*(X.T@(X@w - y))/n
    return w
w_gd0 = ls_gd(Xu, yu)
cos = (w_gd0@w_min)/(np.linalg.norm(w_gd0)*np.linalg.norm(w_min))
print(f'GD-from-0 vs 最小范数: cos={cos:.4f}, ||GD||={np.linalg.norm(w_gd0):.4f}, ||min||={np.linalg.norm(w_min):.4f}')
assert cos > 0.999, 'GD-from-0 应收敛到最小范数方向'
assert abs(np.linalg.norm(w_gd0)-np.linalg.norm(w_min)) < 0.05, '范数应相等'
print('✅ GD-from-0 收敛到最小范数插值解 —— 隐式 min-norm 偏置被钉死。')

## 3 · 验证『停留在行空间』机制

最小范数偏置的机制：GD 更新 $w_{t+1}=w_t-\eta X^\top(\cdots)$ 始终在 $X^\top$ 的列空间(行空间)内。验证：GD 解在 $X$ 的**零空间**上的分量 ≈ 0（即它没碰数据未约束的方向）。

In [ ]:
# 零空间投影算子 P_null = I - X^T (X X^T)^{-1} X
P_null = np.eye(d) - Xu.T @ np.linalg.solve(Xu@Xu.T, Xu)
null_comp = np.linalg.norm(P_null @ w_gd0)
print(f'GD 解在零空间的分量范数 = {null_comp:.2e} (应≈0)')
assert null_comp < 1e-3, 'GD-from-0 不应有零空间分量(只在行空间移动)'
# 对比：一个一般插值解(加任意零空间分量)有非零零空间分量
v = P_null @ rng.standard_normal(d)
w_other = w_min + v
assert np.allclose(Xu@w_other, yu), '加零空间分量仍插值'
assert np.linalg.norm(w_other) > np.linalg.norm(w_min), '但范数更大(非最小范数)'
print(f'另一插值解(加零空间分量)范数 = {np.linalg.norm(w_other):.4f} > 最小范数 {np.linalg.norm(w_min):.4f}')
print('✅ 机制确认：GD-from-0 只在行空间移动 -> 自动得到最小范数解。')

## 4 · SGD 噪声偏向平坦极小

构造一个 1D 双井损失：一个**尖锐**井（窄、深）和一个**平坦**井（宽）。从尖锐井附近出发，对比大噪声 vs 小噪声 SGD 最终停在哪。大噪声更常逃离尖锐井、停在平坦井。

In [ ]:
# 双井: 平坦井在 x=-2(宽), 尖锐井在 x=+2(窄)。用分段二次构造已知曲率。
def loss(x):
    flat  = 0.5*0.2*(x+2)**2          # 曲率 0.2 (平坦)
    sharp = 0.5*5.0*(x-2)**2 - 0.3    # 曲率 5.0 (尖锐), 略深
    return np.minimum(flat, sharp)
def grad(x):
    flat  = 0.5*0.2*(x+2)**2; sharp = 0.5*5.0*(x-2)**2 - 0.3
    return 0.2*(x+2) if flat <= sharp else 5.0*(x-2)
def sgd_well(noise, x0=2.0, steps=4000, lr=0.05, seed=0):
    g = np.random.default_rng(seed); x = x0
    for _ in range(steps):
        x = x - lr*(grad(x) + g.standard_normal()*noise)
    return x
# 多次重复, 统计停在平坦井(x<0)的比例
def frac_flat(noise, reps=200):
    return np.mean([sgd_well(noise, seed=s) < 0 for s in range(reps)])
f_small = frac_flat(0.3); f_large = frac_flat(3.0)
print(f'停在平坦井的比例: 小噪声={f_small:.2f}, 大噪声={f_large:.2f}')
assert f_large > f_small, '大噪声 SGD 应更常逃到平坦井'
print('✅ 大噪声 SGD 偏向平坦极小 —— 噪声(大lr/小batch)的隐式平坦正则。')

## 5 · 平坦度度量 + early stopping ≈ L2 正则

(a) 平坦度：用『损失对随机参数扰动的平均涨幅』度量，验证平坦井 < 尖锐井。
(b) early stopping：最小二乘 GD 第 $t$ 步的解范数随 $t$ 单调增长——早停 = 限制范数 = 隐式 L2。

In [ ]:
# (a) 平坦度 = E_xi[loss(x+xi) - loss(x)], xi~N(0, eps^2)
def sharpness(x, eps=0.3, reps=4000, seed=0):
    g = np.random.default_rng(seed)
    return np.mean([loss(x + g.standard_normal()*eps) - loss(x) for _ in range(reps)])
s_flat = sharpness(-2.0); s_sharp = sharpness(2.0)
print(f'平坦度(扰动涨幅): 平坦井={s_flat:.4f}, 尖锐井={s_sharp:.4f}')
assert s_sharp > s_flat, '尖锐井对扰动更敏感(平坦度更大)'
# (b) early stopping: GD 解范数随训练步数增长
def ls_gd_norm_at(steps, lr=0.05):
    w = np.zeros(d)
    for _ in range(steps): w = w - lr*(Xu.T@(Xu@w - yu))/n
    return np.linalg.norm(w)
norms = [ls_gd_norm_at(s) for s in [5, 20, 100, 1000]]
print('GD 解范数 @ steps[5,20,100,1000] =', [f'{x:.4f}' for x in norms])
assert norms[0] < norms[1] < norms[2] <= norms[3] + 1e-12, '解范数随训练步数单调不减'
assert norms[-1] > norms[0], '范数总体增长(从小到接近最小范数)'
print('✅ 范数随训练增长 -> 早停=限制范数=隐式 L2 正则；平坦度量确认。')

---
## ✏️ 练习 1：最小范数解

实现 `min_norm_solution(X, y)` 返回过参数化（$n<d$）最小二乘的最小范数插值解 $X^\top(XX^\top)^{-1}y$。

In [ ]:
def min_norm_solution(X, y):
    # TODO: 返回 X^T (X X^T)^{-1} y (用 np.linalg.solve)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
Xa = rng.standard_normal((4, 12)); ya = rng.standard_normal(4)
w = min_norm_solution(Xa, ya)
assert np.allclose(Xa @ w, ya), '应插值'
# 最小范数性质: 任意零空间扰动都使范数变大
Pn = np.eye(12) - Xa.T @ np.linalg.solve(Xa@Xa.T, Xa)
v = Pn @ rng.standard_normal(12)
assert np.linalg.norm(w + 0.5*v) >= np.linalg.norm(w) - 1e-9, '应是最小范数'
print(f'最小范数解 ||w||={np.linalg.norm(w):.4f}, 插值残差={np.linalg.norm(Xa@w-ya):.2e}')
print('✅ 练习 1 通过：最小范数插值解（GD-from-0 的极限）')

## ✏️ 练习 2：margin 收敛 —— GD 方向趋于 max-margin

实现 `gd_direction_cos(steps_list)`：对每个 step 数跑 logistic GD，返回其方向与 hard-SVM 方向的 cosine 列表。验证 cosine 随训练步数**单调增**（方向越来越接近 max-margin）。复用 worked 1 的 `logistic_gd`、`hard_svm`、`X`、`y`。

In [ ]:
def gd_direction_cos(steps_list):
    # TODO: w_svm = hard_svm(X,y); 对每个 steps: w=logistic_gd(X,y,steps=steps)
    #       记录 cos(w, w_svm); 返回 cosine 列表
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
coss = gd_direction_cos([200, 2000, 20000])
print('cos @ steps[200,2000,20000] =', [f'{c:.4f}' for c in coss])
assert coss[0] < coss[1] < coss[2], 'cosine 应随训练步数单调增(趋向 max-margin)'
assert coss[-1] > 0.9, '足够多步后应很接近 max-margin'
print('✅ 练习 2 通过：GD 方向随训练单调趋向 max-margin (收敛但慢, ~1/log t)')

## ✏️ 练习 3：平坦度度量

实现 `flatness(loss_fn, x, eps, reps)` 返回损失在 $x$ 处对 $\mathcal N(0,\varepsilon^2)$ 扰动的平均涨幅 $\mathbb E_\xi[\text{loss}(x+\xi)-\text{loss}(x)]$。值越大越尖锐。

In [ ]:
def flatness(loss_fn, x, eps=0.3, reps=4000, seed=0):
    # TODO: 平均 loss_fn(x + N(0,eps)) - loss_fn(x)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
q_flat = flatness(loss, -2.0); q_sharp = flatness(loss, 2.0)
print(f'平坦度: 平坦井={q_flat:.4f}, 尖锐井={q_sharp:.4f}')
assert q_sharp > q_flat > 0, '尖锐井平坦度更大(对扰动更敏感)'
# 平坦度应随扰动尺度 eps 增大
assert flatness(loss, 2.0, eps=0.5) > flatness(loss, 2.0, eps=0.1), '扰动越大涨幅越大'
print('✅ 练习 3 通过：平坦度量化(扰动涨幅), 尖锐井>平坦井')

## ✏️ 练习 4：隐式正则 —— GD 范数路径

实现 `norm_path(steps_list, lr)`：对每个 step 数跑最小二乘 GD-from-0，返回解范数列表（复用 worked 5 的 `Xu,yu,n,d`）。验证范数单调增且上界为最小范数解的范数。

In [ ]:
def norm_path(steps_list, lr=0.05):
    # TODO: 对每个 steps 从 w=0 跑最小二乘 GD, 返回 ||w|| 列表
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
path = norm_path([5, 20, 100, 50000])
wmin_norm = np.linalg.norm(Xu.T @ np.linalg.solve(Xu@Xu.T, yu))
print('范数路径:', [f'{x:.4f}' for x in path], ' -> 最小范数 =', f'{wmin_norm:.4f}')
assert path[0] < path[1] < path[2], '范数随训练单调增'
assert abs(path[-1] - wmin_norm) < 1e-2, '范数收敛到最小范数解'
print('✅ 练习 4 通过：范数沿 GD 路径增长并收敛到最小范数 -> 早停=隐式 L2')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1
def min_norm_solution(X, y):
    return X.T @ np.linalg.solve(X @ X.T, y)

In [ ]:
# 练习 2
def gd_direction_cos(steps_list):
    w_svm = hard_svm(X, y)
    out = []
    for s in steps_list:
        w = logistic_gd(X, y, steps=s)
        out.append((w@w_svm)/(np.linalg.norm(w)*np.linalg.norm(w_svm)))
    return out

In [ ]:
# 练习 3
def flatness(loss_fn, x, eps=0.3, reps=4000, seed=0):
    g = np.random.default_rng(seed)
    return np.mean([loss_fn(x + g.standard_normal()*eps) - loss_fn(x) for _ in range(reps)])

In [ ]:
# 练习 4
def norm_path(steps_list, lr=0.05):
    out = []
    for s in steps_list:
        w = np.zeros(d)
        for _ in range(s): w = w - lr*(Xu.T@(Xu@w - yu))/n
        out.append(np.linalg.norm(w))
    return out

---
## 🧪 真实数据胶囊：真实数据上的隐式 margin 最大化

用真实数据（sklearn 乳腺癌取 6 个特征；失败回退到合成可分数据）验证 logistic GD 在真实数据上也持续**增大间隔**。注意：6 特征子集**未必线性可分**，所以最小归一化 margin 可能为负（有点被分错）——但关键是训练越久，这个 margin **越大**（朝 max-margin 方向走），训练准确率也高。这正是隐式偏置在真实数据上的体现。

In [ ]:
try:
    from sklearn.datasets import load_breast_cancer
    dd = load_breast_cancer()
    Xc = dd.data.astype(float); yc = (dd.target*2-1).astype(float)
    Xc = (Xc - Xc.mean(0))/(Xc.std(0)+1e-9)
    # 取一个低维子空间(前6主成分方向的列)使其更易线性可分, 并单位化
    Xc = Xc[:, :6]
    src = 'sklearn breast_cancer (真实, 6 特征)'
except Exception as e:
    Xc, yc = make_separable(200, 6, seed=1, margin=0.5); src = f'回退合成可分: {type(e).__name__}'
print(f'数据来源: {src}; 形状 {Xc.shape}')

In [ ]:
def logistic_gd_general(X, y, steps=40000, lr=0.5):
    n, d = X.shape; w = np.zeros(d)
    for _ in range(steps):
        p = 1/(1+np.exp(y*(X@w))); w = w + lr*(X.T@(p*y))/n
    return w
w_real = logistic_gd_general(Xc, yc, steps=20000)
acc = np.mean(np.sign(Xc@w_real) == yc)
nm = (yc*(Xc@w_real)).min()/np.linalg.norm(w_real)   # 最小归一化 margin
print(f'真实数据 logistic GD: 训练准确率={acc:.3f}, 最小归一化 margin={nm:.4f}')
print('(margin 可能为负: 6 特征子集未必完全可分; 关键看训练更久 margin 是否增大 -> 见胶囊练习)')
assert acc > 0.85, '真实数据上应分得不错'
print('✅ 真实数据上 GD 同样朝大间隔方向走（训练准确率高）')

**🧪 胶囊练习**：实现 `margin_grows(steps_a, steps_b)` 返回 `(margin@steps_a, margin@steps_b)`——验证训练更久最小归一化 margin 不减（趋于 max-margin）。

In [ ]:
def margin_grows(steps_a, steps_b):
    # TODO: 对两个步数各跑 logistic_gd_general, 返回各自的最小归一化 margin
    #       (y*(Xc@w)).min()/||w||
    raise NotImplementedError

In [ ]:
# 胶囊自测
def margin_grows(steps_a, steps_b):
    def m(s):
        w = logistic_gd_general(Xc, yc, steps=s)
        return (yc*(Xc@w)).min()/np.linalg.norm(w)
    return m(steps_a), m(steps_b)
ma, mb = margin_grows(2000, 20000)
print(f'最小归一化 margin: 2000步={ma:.4f}, 20000步={mb:.4f}')
assert mb >= ma - 1e-3, '训练更久 margin 不应变差(趋于 max-margin)'
print('✅ 胶囊练习通过：训练越久归一化 margin 越大(趋于 max-margin)')

In [ ]:
# 📖 胶囊参考答案
def margin_grows(steps_a, steps_b):
    def m(s):
        w = logistic_gd_general(Xc, yc, steps=s)
        return (yc*(Xc@w)).min()/np.linalg.norm(w)
    return m(steps_a), m(steps_b)

---
### 小结
- 过参数化下零损失解有无穷多，GD 隐式偏好『简单』的那个——泛化之谜重新定位到『算法挑了谁』。
- 可分 + logistic：GD 方向 → max-margin(hard-SVM)解（cos→0.9997），收敛慢 ~$1/\log t$。
- 最小二乘 + GD-from-0 → 最小范数插值解（cos=1.0），机制=迭代停留在行空间(零空间分量≈0)。
- SGD 噪声(大lr/小batch)隐式偏向平坦极小；但平坦度非参数化无关([Dinh 2017] 警告)。
- early stopping ≈ L2 正则（解范数随训练单调增并收敛到最小范数）。

下一站：**模块 05 · 深度泛化**——把这些隐式偏好*量化*成界（margin、PAC-Bayes），直面 Zhang 随机标签难题。